In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

def selecionar_arquivo(entry_widget):
    """Abre a janela para escolher o arquivo Excel correspondente"""
    arquivo = filedialog.askopenfilename(
        title="Selecione a planilha",
        filetypes=[("Arquivos Excel", "*.xlsx *.xls")]
    )
    if arquivo:
        entry_widget.delete(0, tk.END)
        entry_widget.insert(0, arquivo)


def executar_auditoria():
    caminho_injetaq = entry_injetaq.get()
    caminho_icms = entry_icms.get()
    caminho_ipi = entry_ipi.get()
    nome_saida = entry_saida.get().strip()

    # Validações iniciais
    if not caminho_injetaq or not caminho_icms or not caminho_ipi:
        messagebox.showwarning("Atenção", "Por favor, selecione os três arquivos de entrada!")
        return
    
    if not nome_saida:
        nome_saida = "resultado_auditoria.xlsx"
    elif not nome_saida.endswith(".xlsx"):
        nome_saida += ".xlsx"

    try:
        # 1. Leitura das planilhas
        df_injetaq = pd.read_excel(caminho_injetaq)
        df_kato_icms = pd.read_excel(caminho_icms)
        df_kato_ipi = pd.read_excel(caminho_ipi)

        # 2. Padronização e criação do id_unico
        # 1. Descobre o nome real da coluna de Número e de Natureza na tabela da empresa
        col_num_inj = encontrar_coluna(df_injetaq, ['Número', 'Numero', 'NF', 'Nota', 'Nº'])
        col_nat_inj = encontrar_coluna(df_injetaq, ['Natureza', 'CFOP', 'Nat'])

        if not col_num_inj or not col_nat_inj:
            raise Exception(f"Não foi possível encontrar a coluna de Número ou Natureza no arquivo da Empresa. Colunas encontradas: {list(df_injetaq.columns)}")

        df_injetaq['id_unico'] = df_injetaq[col_num_inj].astype(str) + '|' + df_injetaq[col_nat_inj].astype(str)

        # Faça o mesmo para as outras tabelas (ICMS e IPI):
        col_num_icms = encontrar_coluna(df_kato_icms, ['Número', 'Numero', 'NF', 'Nota', 'Nº'])
        col_nat_icms = encontrar_coluna(df_kato_icms, ['Natureza', 'CFOP', 'Nat'])
        df_kato_icms['id_unico'] = df_kato_icms[col_num_icms].astype(str) + '|' + df_kato_icms[col_nat_icms].astype(str)

        col_num_ipi = encontrar_coluna(df_kato_ipi, ['Número', 'Numero', 'NF', 'Nota', 'Nº'])
        col_nat_ipi = encontrar_coluna(df_kato_ipi, ['Natureza', 'CFOP', 'Nat'])
        df_kato_ipi['id_unico'] = df_kato_ipi[col_num_ipi].astype(str) + '|' + df_kato_ipi[col_nat_ipi].astype(str)

        # 3. Renomeia as colunas de valor para evitar conflitos de nomes
        df_injetaq = df_injetaq.rename(columns={
            'Valor Contábil': 'Contabil_Injetaq',
            'Valor Icms': 'Icms_Injetaq',
            'Valor IPI': 'Ipi_Injetaq'
        })
        df_kato_icms = df_kato_icms.rename(columns={
            'Vlr Contábil Kato ICMS': 'Contabil_ICMS',
            'Valor ICMS Kato': 'Icms_ICMS'
        })
        df_kato_ipi = df_kato_ipi.rename(columns={
            'vl_cont_ipi_kato': 'Contabil_IPI',
            'vl_ipi_kato': 'Ipi_IPI'
        })

        # 4. Agrupamento para somar notas iguais na mesma natureza (evitando duplicações)
        df_injetaq_g = df_injetaq.groupby('id_unico', as_index=False).sum(numeric_only=True)
        df_icms_g = df_kato_icms.groupby('id_unico', as_index=False).sum(numeric_only=True)
        df_ipi_g = df_kato_ipi.groupby('id_unico', as_index=False).sum(numeric_only=True)

        # 5. Criação da Lista Mestre
        lista_mestre = pd.concat([
            df_injetaq_g['id_unico'],
            df_icms_g['id_unico'],
            df_ipi_g['id_unico']
        ]).dropna().drop_duplicates()

        df_mestre = pd.DataFrame({'id_unico': lista_mestre})

        # 6. Merge (Left Join)
        df_mestre = pd.merge(df_mestre, df_injetaq_g, on='id_unico', how='left')
        df_mestre = pd.merge(df_mestre, df_icms_g, on='id_unico', how='left')
        df_mestre = pd.merge(df_mestre, df_ipi_g, on='id_unico', how='left')

        # 7. Limpeza imediata dos vazios (NaN) com 0 para evitar erros nas diferenças
        df_mestre = df_mestre.fillna(0)

        # 8. Divisão do ID em 'nf' e 'natureza' e conversão para inteiros limpos
        df_mestre[['nf', 'natureza']] = df_mestre['id_unico'].str.split('|', expand=True)
        df_mestre['nf'] = pd.to_numeric(df_mestre['nf'], errors='coerce').astype('Int64')
        df_mestre['natureza'] = pd.to_numeric(df_mestre['natureza'], errors='coerce').astype('Int64')

        # 9. Cálculo das Diferenças (Empresa vs Contabilidade correspondente)
        df_mestre['Dif Valor Contabil'] = df_mestre['Contabil_Injetaq'] - df_mestre['Contabil_ICMS']
        df_mestre['Dif Valor ICMS'] = df_mestre['Icms_Injetaq'] - df_mestre['Icms_ICMS']
        df_mestre['Dif Valor IPI'] = df_mestre['Ipi_Injetaq'] - df_mestre['Ipi_IPI']

        # Arredonda as diferenças para 2 casas decimais (eliminando dízimas E-12)
        cols_dif = ['Dif Valor Contabil', 'Dif Valor ICMS', 'Dif Valor IPI']
        for col in cols_dif:
            df_mestre[col] = df_mestre[col].round(2)

        # 10. Classificação (OK ou Valor Divergente)
        df_mestre['Classificacao'] = 'Valor Divergente'
        df_mestre.loc[df_mestre[cols_dif].eq(0).all(axis=1), 'Classificacao'] = 'OK'

        # 11. Organização das colunas principais no início
        cols_principais = ['id_unico', 'nf', 'natureza', 'Classificacao']
        outras_cols = [c for c in df_mestre.columns if c not in cols_principais]
        df_mestre = df_mestre[cols_principais + outras_cols]

        # 12. Criação da Aba de Resumo Executivo (Indicadores nas colunas, valores nas linhas)
        qtde_nf_empresa = (df_mestre['Contabil_Injetaq'] != 0).sum()
        qtde_nf_icms_kato = (df_mestre['Contabil_ICMS'] != 0).sum()

        valor_contabil_empresa = df_mestre['Contabil_Injetaq'].sum()
        valor_contabil_icms = df_mestre['Contabil_ICMS'].sum()
        
        valor_icms_empresa = df_mestre['Icms_Injetaq'].sum()
        valor_icms_kato = df_mestre['Icms_ICMS'].sum()

        valor_ipi_empresa = df_mestre['Ipi_Injetaq'].sum()
        valor_ipi_kato = df_mestre['Ipi_IPI'].sum()

        df_resumo = pd.DataFrame({
            'Qtde Nfs Empresa': [qtde_nf_empresa],
            'Qtde Nfs Contabilidade': [qtde_nf_icms_kato],
            'Diferença qtde Nfs': [qtde_nf_empresa - qtde_nf_icms_kato],
            'Valor Total Contabil Empresa': [valor_contabil_empresa],
            'Valor Total Contabil Contabilidade': [valor_contabil_icms],
            'Diferença valor total Contabil': [valor_contabil_empresa - valor_contabil_icms],
            'Valor Total ICMS Empresa': [valor_icms_empresa],
            'Valor Total ICMS Contabilidade': [valor_icms_kato],
            'Diferença Valor total ICMS': [valor_icms_empresa - valor_icms_kato],
            'Valor Total IPI Empresa': [valor_ipi_empresa],
            'Valor Total IPI Contabilidade': [valor_ipi_kato],
            'Diferença Valor total IPI': [valor_ipi_empresa - valor_ipi_kato]
        })

        # 13. Pergunta ao usuário onde deseja salvar o arquivo final
        caminho_salvar = filedialog.asksaveasfilename(
            defaultextension=".xlsx",
            initialfile=nome_saida,
            filetypes=[("Arquivo Excel", "*.xlsx")]
        )

        if caminho_salvar:
            # Salva o Excel com as duas abas
            with pd.ExcelWriter(caminho_salvar, engine='openpyxl') as writer:
                df_mestre.to_excel(writer, sheet_name='Auditoria_Detalhada', index=False)
                df_resumo.to_excel(writer, sheet_name='Resumo_Executivo', index=False)

            # Reabre para aplicar as cores verde/vermelho na aba detalhada
            wb = load_workbook(caminho_salvar)
            ws = wb['Auditoria_Detalhada']

            fill_verde = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
            fill_vermelho = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")

            # Descobre a coluna 'Classificacao'
            col_idx_status = None
            for col_i, cell in enumerate(ws[1], 1):
                if cell.value == 'Classificacao':
                    col_idx_status = col_i
                    break

            if col_idx_status:
                for row in range(2, ws.max_row + 1):
                    cell = ws.cell(row=row, column=col_idx_status)
                    if cell.value == 'OK':
                        cell.fill = fill_verde
                    elif cell.value == 'Valor Divergente':
                        cell.fill = fill_vermelho

            wb.save(caminho_salvar)
            messagebox.showinfo("Sucesso!", f"Auditoria concluída e salva com sucesso em:\n{caminho_salvar}")

    except Exception as e:
        messagebox.showerror("Erro no Processamento", f"Ocorreu um erro durante a execução:\n{str(e)}")

# --- CONSTRUÇÃO DA TELA (GUI) ---
root = tk.Tk()
root.title("Ferramenta de Auditoria Fiscal - Contabilidade")
root.geometry("600x420")
root.resizable(True, True)

tk.Label(root, text="Automação de Cruzamento Fiscal", font=("Arial", 14, "bold")).pack(pady=15)

# Campo 1: Injetaq
tk.Label(root, text="Livro de Entradas (Injetaq / Empresa):", anchor="w", font=("Arial", 9, "bold")).pack(fill="x", padx=25)
f1 = tk.Frame(root)
f1.pack(fill="x", padx=25, pady=3)
entry_injetaq = tk.Entry(f1, width=60)
entry_injetaq.pack(side="left", padx=(0, 5))
tk.Button(f1, text="Procurar", command=lambda: selecionar_arquivo(entry_injetaq)).pack(side="left")

# Campo 2: ICMS
tk.Label(root, text="Livro de ICMS (Contabilidade):", anchor="w", font=("Arial", 9, "bold")).pack(fill="x", padx=25, pady=(8,0))
f2 = tk.Frame(root)
f2.pack(fill="x", padx=25, pady=3)
entry_icms = tk.Entry(f2, width=60)
entry_icms.pack(side="left", padx=(0, 5))
tk.Button(f2, text="Procurar", command=lambda: selecionar_arquivo(entry_icms)).pack(side="left")

# Campo 3: IPI
tk.Label(root, text="Livro de IPI (Contabilidade):", anchor="w", font=("Arial", 9, "bold")).pack(fill="x", padx=25, pady=(8,0))
f3 = tk.Frame(root)
f3.pack(fill="x", padx=25, pady=3)
entry_ipi = tk.Entry(f3, width=60)
entry_ipi.pack(side="left", padx=(0, 5))
tk.Button(f3, text="Procurar", command=lambda: selecionar_arquivo(entry_ipi)).pack(side="left")

# Campo 4: Nome do arquivo final
tk.Label(root, text="Nome do Relatório de Saída (.xlsx):", anchor="w", font=("Arial", 9, "bold")).pack(fill="x", padx=25, pady=(8,0))
entry_saida = tk.Entry(root, width=67)
entry_saida.pack(padx=25, pady=3, anchor="w")
entry_saida.insert(0, "auditoria_fiscal_resultado.xlsx")

# Botão Executar Principal
btn_executar = tk.Button(
    root, 
    text="Executar Auditoria e Gerar Excel", 
    bg="#2e7d32", 
    fg="white", 
    font=("Arial", 11, "bold"), 
    padx=10, 
    pady=5,
    command=executar_auditoria
)
btn_executar.pack(pady=20)

root.mainloop()